In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pathlib import Path
import re
from data_profiling import ProfileReport
import platform
from utils import apply_filters_to_df

In [19]:
if platform.system() == 'Windows':
    netapp_dir = Path(r"W:/MinMo_CI/")
else:
    netapp_dir = Path.home() / "Documents/Postdoc_epilepsy/MinMo_CI"

In [20]:
TARGET_SEQUENCE_NAME =  "*tse2d1_17"
TARGET_SEQUENCE_NAME = None
# load the NGS results and DICOM metadata csv files
suffix = f"_{TARGET_SEQUENCE_NAME.replace('*', '')}" if TARGET_SEQUENCE_NAME else "_all"
dicom_metadata_df = pd.read_csv(netapp_dir / "derivatives" / f"DICOM_inventory{suffix}.csv")
json_metadata_df = pd.read_csv(netapp_dir / "derivatives" / f"JSON_metadata_inventory{suffix}.csv")
ngs_results_df = pd.read_csv(netapp_dir / "derivatives" / f"NGS_results{suffix}.csv")
# load the group key csv file to get the group labels (minmo or not) and age for each subject
group_key_df = pd.read_csv(netapp_dir / "derivatives" / "group_key.csv")

In [21]:
# clean the SubjectID column in the dataframes to match the format across all dataframes (replace 'MinMo-' with 'Min-Mo-' and strip whitespace)
group_key_df['SubjectID'] = group_key_df['SubjectID'].str.replace('Min-Mo-', 'MinMo-').str.strip()
json_metadata_df['SubjectID'] = json_metadata_df['SubjectID'].str.replace('Min-Mo-', 'MinMo-').str.strip()
ngs_results_df['SubjectID'] = ngs_results_df['SubjectID'].str.replace('Min-Mo-', 'MinMo-').str.strip()

In [22]:
# create a merge key for both dataframes based on the file path of the nifti file in the derivatives folder, which is unique for each series
json_metadata_df['MergeKey'] = json_metadata_df['NIfTIPath'].apply(lambda x: str(Path(x)))
ngs_results_df['MergeKey'] = ngs_results_df['FilePath'].apply(lambda x: str(Path(x)))

In [23]:
# merge on series hash which is the file path of the nifti file in the derivatives folder, which is unique for each series
merged_df = ngs_results_df.merge(json_metadata_df.drop(columns=['NIfTIPath', 'SubjectID']), on='MergeKey', how='inner')

In [24]:
# merge the merged dataframe with the group key dataframe to get the group labels for each subject
merged_df = merged_df.merge(group_key_df, on='SubjectID', how='left')
len(merged_df)

903

In [25]:
space_df = merged_df[merged_df['PulseSequenceName'].str.contains('spcR', na=False)]
print(space_df.groupby('PulseSequenceName')[['EchoTime','RepetitionTime','FlipAngle','MagneticFieldStrength','SliceThickness','EchoTrainLength']].mean())
flair_df = merged_df[merged_df['PulseSequenceName'].str.contains('spcir', na=False)]
print(flair_df.groupby('PulseSequenceName')[['EchoTime','RepetitionTime','FlipAngle','MagneticFieldStrength','SliceThickness','EchoTrainLength']].mean())


                   EchoTime  RepetitionTime  FlipAngle  MagneticFieldStrength  \
PulseSequenceName                                                               
*spcR_40ns         0.007267         0.60000      120.0                    1.5   
*spcR_42ns         0.022004         0.69031      120.0                    3.0   
*spcR_50ns         0.022000         0.70000      120.0                    3.0   

                   SliceThickness  EchoTrainLength  
PulseSequenceName                                   
*spcR_40ns               1.734177             40.0  
*spcR_42ns               1.751575             27.0  
*spcR_50ns               1.799999             35.0  
                   EchoTime  RepetitionTime  FlipAngle  MagneticFieldStrength  \
PulseSequenceName                                                               
*spcir_210ns       0.368816             7.0      120.0                    1.5   
*spcir_220ns       0.392000             7.0      120.0                    3.0   

    

In [26]:
# add QC flags and other useful columns to the merged dataframe
merged_df['QC_flag'] = merged_df['Num_voxels_in_mask_Eroded'] < 1000  # flag if fewer than 1000 brain voxels after erosion
merged_df['QC_flag_2'] = merged_df['Num_voxels_in_mask_Eroded'] / merged_df['Num_voxels_in_mask'] < 0.5  # flag if fewer than 50% of brain voxels remain after erosion  
merged_df['IsSWI'] = merged_df['PulseSequenceName'].str.contains('swi', case=False, na=False)
merged_df['IsspcR'] = merged_df['PulseSequenceName'].str.contains('spcR', case=False, na=False)
merged_df['Isspcir'] = merged_df['PulseSequenceName'].str.contains('spcir', case=False, na=False)
merged_df['IsPhase'] = merged_df['FilePath'].str.contains('_ph.nii.gz', na=False)
merged_df['IsMIP'] = merged_df['FilePath'].str.contains('mIP|MIP', na=False)
merged_df['Is3D'] = merged_df['MRAcquisitionType'] == '3D'
merged_df['isAnat'] = merged_df['BidsGuess'].str.contains('anat', case=False, na=False) 
merged_df['isDerived'] = merged_df['BidsGuess'].str.contains('derived', case=False, na=False)
merged_df['isFunc'] = merged_df['BidsGuess'].str.contains('func', case=False, na=False)

In [27]:
cols_to_profile = ['SubjectID', 'PulseSequenceName', 'Orientation', 'MagneticFieldStrength', 
                   'MRAcquisitionType', 'SliceThickness', 'EchoTime', 'RepetitionTime',
                   'IsSWI','Isspcir', 'IsspcR', 'IsPhase', 'IsMIP', 'Is3D','Mean_NGS_Eroded',
                   'Q25_NGS_Eroded', 'Q75_NGS_Eroded',
                   'Median_NGS_Eroded', 'Std_NGS_Eroded', 'N_slices', 
                   'Num_voxels_in_mask_Eroded', 'QC_flag', 'QC_flag_2', 
                   'isAnat', 'isDerived', 'isFunc', 'MinMo_Group', 'Age at scan']
profile = ProfileReport(merged_df[cols_to_profile], title="NGS Results Profiling Report", explorative=True)
profile.to_file(netapp_dir / "derivatives" / f"NGS_results_profiling_report{suffix}.html")

c:\Users\jm19\MinMo_CI\.venv\Lib\site-packages\data_profiling\utils\dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 28/28 [00:00<00:00, 273754.11it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [28]:
# save the merged dataframe to a new csv file 
merged_csv_file = netapp_dir / "derivatives" / f"NGS_json_merged{suffix}.csv"
merged_df.to_csv(merged_csv_file, index=False)
print(f"Merged NGS and json metadata saved to {merged_csv_file}")

Merged NGS and json metadata saved to W:\MinMo_CI\derivatives\NGS_json_merged_all.csv


In [ ]:
from utils import apply_filters_to_df
#filtered_df = merged_df[(merged_df['Orientation'] == 'axial') & (~merged_df['FilePath'].str.contains('Med_DRS_480'))].copy()
# example call: apply_filters_to_df(merged_df, {'Orientation': 'axial', 'FilePath': ('excludes', 'Med_DRS_480')})
filtered_df = apply_filters_to_df(merged_df, {'BidsGuess': ('contains', "'anat'"), 'FilePath': ('excludes', 't1post'), 'FilePath': ('excludes', '_ph.nii.gz'), 'FilePath': ('excludes', '_mIP.nii.gz')})


In [ ]:
#axial_df['QC_scanner'] = axial_df['MagneticFieldStrength'] != 3
print(f"Total rows before dedup: {len(filtered_df)}")
filtered_df = filtered_df.sort_values('SeriesNumber').drop_duplicates(subset='SubjectID', keep='first')
print(f"Total rows after dedup: {len(filtered_df)}")

filtered_df_clean = filtered_df[~filtered_df['QC_flag'] & ~filtered_df['QC_flag_2']].copy()
print(f"Clean subjects: {len(filtered_df_clean)}")
print(filtered_df_clean['MinMo_Group'].value_counts())

# save the filtered dataframe to a new csv file and generate a profiling report
filtered_csv_file = netapp_dir / "derivatives" / f"Filtered_NGS_results{suffix}.csv"
filtered_df_clean.to_csv(filtered_csv_file, index=False)
print(f"Filtered NGS metadata saved to {filtered_csv_file}")
ProfileReport(filtered_df_clean, title=f"Filtered NGS {len(filtered_df_clean)} Profiling Report", explorative=True).to_file(netapp_dir / "derivatives" / f"Filtered_NGS_results_profiling_report{suffix}.html")

In [ ]:
%matplotlib inline
plt.figure(figsize=(5, 3))
sns.kdeplot(data=filtered_df_clean, x='Median_NGS_Eroded', hue='MinMo_Group', fill=True, alpha=0.5)
plt.title('Distribution of NGS by MinMo Group')
plt.xlabel('Median NGS (Eroded Mask)')
plt.savefig(netapp_dir / "derivatives" / "NGS_distribution_by_group.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(filtered_df_clean['MagneticFieldStrength'].value_counts())
print(filtered_df_clean.groupby(['MagneticFieldStrength'])['Median_NGS_Eroded'].describe())
print(filtered_df_clean.groupby(['MinMo_Group','MagneticFieldStrength']).size().unstack(fill_value=0))

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='MinMo_Group', y='Median_NGS_Eroded', data=filtered_df_clean, showfliers=False)
plt.title('NGS by MinMo Group')
plt.xlabel('Group')
plt.ylabel('Median NGS (Eroded Mask)')
plt.ylim(0, 0.0002)
plt.savefig(netapp_dir / "derivatives" / "NGS_boxplot_minmo_groups.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from scipy.stats import mannwhitneyu, spearmanr, shapiro
import statsmodels.formula.api as smf

with_minmo = filtered_df_clean[filtered_df_clean['MinMo_Group'] == 'With MinMo']['Median_NGS_Eroded']
without_minmo = filtered_df_clean[filtered_df_clean['MinMo_Group'] == 'Without MinMo']['Median_NGS_Eroded']

stat, p = mannwhitneyu(with_minmo, without_minmo, alternative='two-sided')
stat_with, p_with = shapiro(with_minmo)
stat_without, p_without = shapiro(without_minmo)

# shapiro-wilk test for normality in both groups i.e. With MinMo and Without MinMo
print(f"Shapiro-Wilk With Minmo: stat={stat_with:.3f}, p={p_with:.3f}")
print(f"Shapiro-Wilk Without Minmo: stat_without={stat_without:.3f}, p={p_without:.3f}")

# mann-whitney u test for difference in distributions between the two groups  i.e. With MinMo and Without MinMo
print(f"Mann-Whitney U statistic: {stat:.3f}, p-value: {p:.3f}")

# multiple regression analysis
model = smf.ols('Median_NGS_Eroded ~ C(MinMo_Group) + Q("Age at scan") + SeriesNumber + MagneticFieldStrength', data=filtered_df_clean).fit() 
# Q is used to handle column names with spaces
print(f"\n Multiple regression analysis summary:")
print(model.summary())


In [ ]:
filtered_df.groupby('MinMo_Group')['Age at scan'].describe()

In [ ]:
axial_df['NGS_residuals'] = model.resid
plt.figure(figsize=(10, 3))
# plot the raw, residuals and predicted/regressed NGS values by MinMo_Group after adjusting for Age and SeriesNumber in subplots
plt.subplot(1, 3, 1)
sns.boxplot(x='MinMo_Group', y='Median_NGS_Eroded', data=axial_df)
plt.title('Raw NGS')
plt.ylabel('Median NGS (Eroded Mask)')
plt.subplot(1, 3, 2)
sns.boxplot(x='MinMo_Group', y=model.fittedvalues, data=axial_df)
plt.title('Predicted NGS\n(SeriesNumber + Age)')
plt.ylabel('Predicted NGS (Eroded Mask)')
plt.subplot(1, 3, 3)
sns.boxplot(x='MinMo_Group', y='NGS_residuals', data=axial_df)
plt.axhline(y =0, color='black', linestyle='--') 
plt.title('Residuals\n(SeriesNumber + Age controlled)') 
plt.ylabel('Residuals of Median NGS ')
plt.tight_layout()
plt.show()

In [ ]:
# correlation between Age at scan and Median_NGS_Eroded
r1, p1 = spearmanr(axial_df['Age at scan'], axial_df['Median_NGS_Eroded'])
print(f"Spearman correlation between Age at scan and Median_NGS_Eroded: Spearman r={r1:.3f}, p={p1:.6f}")
print(axial_df.groupby('MinMo_Group')['Age at scan'].describe()) # group by MinMo_Group and describe Age at scan

# correlation between SeriesNumber and Median_NGS_Eroded
r2, p2 = spearmanr(axial_df['SeriesNumber'], axial_df['Median_NGS_Eroded'])
print(f"\nSpearman correlation between SeriesNumber and Median_NGS_Eroded: Spearman r={r2:.3f}, p={p2:.6f}")
print(axial_df.groupby('MinMo_Group')['SeriesNumber'].describe()) # group by MinMo_Group and describe SeriesNumber


In [ ]:
plt.figure(figsize=(10,6))
sns.regplot(x='Num_voxels_in_mask_Eroded', y='Mean_NGS_Eroded', data=axial_df)
plt.axvline(x=1000, color='red', linestyle='--', label='QC Threshold')
plt.title('Mean NGS  vs number of voxels in eroded Mask')
plt.xlabel('Number of voxels in eroded mask')
plt.ylabel('Mean NGS')
plt.legend()
#plt.savefig(netapp_dir / "derivatives" / "NGS_vs_mask_voxels.png", dpi=150, bbox_inches='tight')
#plt.close()
plt.show()

In [ ]:
json_metadata_all_df = pd.read_csv(netapp_dir / "derivatives" / "JSON_metadata_inventory_all.csv")
clean_df = json_metadata_all_df[json_metadata_all_df['BidsGuess'].str.contains("'anat'") & # include only anatomical images
                                ~json_metadata_all_df['NIfTIPath'].str.contains('post', case=False) & #exclude post-contrast images
                                ~json_metadata_all_df['NIfTIPath'].str.contains('_ph.nii', case=False) # exclude phase images
                                ] 
print(f"Total images after filtering: {len(clean_df)}")
print(clean_df['PulseSequenceName'].value_counts())